In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', module='tqdm')

In [ ]:
data_path = "../raw_data/"
file_name = "Sample_Rotterdam.xlsx"
file_path = data_path + file_name
file_path

In [ ]:
df = pd.read_excel(file_path)
df.info()

In [ ]:
not_string_count = 0
not_string_list = []

for i in df["Toelichting_unmasked"]:
    if type(i) != str:
        not_string_count += 1
        not_string_list.append(i)
if not_string_count == 0:
    print("All texts are of type String.")
else:
    print(f"There are {not_string_count} texts that are not of type String:")
    print(not_string_list)

In [ ]:
col = "Toelichting_unmasked"          # column you want to check
initial_rows = len(df)                # rows before cleaning

# --- drop rows with NaN in that column ------------------------------
df_clean = df.dropna(subset=[col])

# --- stats ----------------------------------------------------------
final_rows   = len(df_clean)
removed_rows = initial_rows - final_rows
pct_removed  = removed_rows / initial_rows * 100

print(f"Rows before : {initial_rows}")
print(f"Rows after  : {final_rows}")
print(f"Removed     : {removed_rows} ({pct_removed:.2f}%)")

In [ ]:
# Entire dataframe masking

import importlib
from anonymizer import CombinedAnonymizer


# Recommend to set to False for producing masked outputs
distinct_tags = False

an = CombinedAnonymizer(distinct_tags=distinct_tags)
df_clean["Toelichting_masked"] = df_clean["Toelichting_unmasked"].map(an)   # __call__ overload
df_clean.to_excel(f"../results/masked_regex.xlsx", index=False)

In [ ]:
# Generate a testset with samples from each group and their masked versions

from anonymizer.helpers.sample_testset import sample_per_group, build_testset

# ---- parameters ----------------------------------------------------------
OUTPUT_FILE         = "../results/testset.xlsx"
TEXT_COLUMN         = "Toelichting_unmasked"
GROUP_COLUMN        = "Bron_Groepering"
SAMPLE_SIZE         = 10
SEED                = 42
ANONYMIZE_METHOD    = "Combined"
DISTINCT_TAGS       = True # Recommend to set to True for testsets
PRINT_NER_CONFIDENCE= True
# --------------------------------------------------------------------------

# 1. Load the full dataset
df = df_clean

# 2. Draw a reproducible 400-row sample for every group
sampled = sample_per_group(
    df,
    group_col=GROUP_COLUMN,
    n=SAMPLE_SIZE,
    seed=SEED,
)

# 3. Run the fixed-pattern anonymiser
if ANONYMIZE_METHOD == "Regex":
    from anonymizer import RegexAnonymizer
    anonymizer = RegexAnonymizer(distinct_tags=DISTINCT_TAGS)
elif ANONYMIZE_METHOD == "NER":
    from anonymizer import NERAnonymizer
    anonymizer = NERAnonymizer(distinct_tags=DISTINCT_TAGS, print_confidence=PRINT_NER_CONFIDENCE)
elif ANONYMIZE_METHOD == "List":
    from anonymizer import ListAnonymizer
    anonymizer = ListAnonymizer(distinct_tags=DISTINCT_TAGS)
elif ANONYMIZE_METHOD == "Combined":
    from anonymizer import CombinedAnonymizer
    anonymizer = CombinedAnonymizer(distinct_tags=DISTINCT_TAGS, print_confidence=PRINT_NER_CONFIDENCE)
else:
    raise ValueError(f"Unknown ANONYMIZE_METHOD: {ANONYMIZE_METHOD}")

testset_df = build_testset(sampled, TEXT_COLUMN, GROUP_COLUMN, anonymizer)

# 4. Persist the workbook
testset_df.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")

print(f"Wrote {len(testset_df)} rows to {OUTPUT_FILE}")



In [ ]:
# Testing the script on a sample text
import warnings
warnings.filterwarnings('ignore', module='tqdm')

import importlib, anonymizer
anonymizer = importlib.reload(anonymizer)

# from anonymizer import RegexAnonymizer
# from anonymizer import ListAnonymizer
# from anonymizer import NERAnonymizer
from anonymizer import CombinedAnonymizer


# --------------------------------------------------------------------------- #
# Doctest                                                                    #
# --------------------------------------------------------------------------- #
distinct_tags = True
print_NER_confidence = True

# a = RegexAnonymizer(distinct_tags=distinct_tags)
# a = ListAnonymizer(distinct_tags=distinct_tags)
# a = NERAnonymizer(distinct_tags=distinct_tags, print_confidence=print_confidence)
a = CombinedAnonymizer(distinct_tags=distinct_tags, print_confidence=print_NER_confidence)


sample = """
Ik had op 1 oktober om 14:05 uur een afspraak op stadhuis aan balie 8. 
De medewerker genaamd Klaasen was heel onbeschoft. 
Ik wil hiervoor een klacht indienen

"""


print(a(sample))